In [1]:
# Student Grade Prediction - MLFLow


In [13]:
import os
import json
import random
import hashlib
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


import joblib

warnings.filterwarnings("ignore")

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
os.environ["PYTHONHASHSEED"] = str(SEED)

In [2]:
rng = np.random.default_rng(42)

N = 2000

attendence = rng.uniform(40,100,N)

study_hours = rng.uniform(0,8,N)

previous_score = rng.uniform(20,100,N)

assignment_completed = rng.integers(0,11,N)

noise = rng.normal(0,10,N)

preformance_score = (
    0.25*attendence +
    4*study_hours +
    0.35*previous_score +
    2*assignment_completed +
    noise
)

passed = (preformance_score >= 75).astype(int)

df = pd.DataFrame({
    "attendence": attendence,
    "study_hours": study_hours,
    "previous_score": previous_score,
    "assignment_completed": assignment_completed,
    "passed": passed
})

df.head()

,attendence,study_hours,previous_score,assignment_completed,passed
0,86.437363,6.741856,77.287998,5,0
1,66.332706,3.571116,48.569248,5,0
2,91.515875,7.623801,88.149428,0,0
3,81.842082,5.206357,39.278173,1,0
4,45.650641,0.927191,63.071479,6,0


In [3]:
# EDA
# Shape
# IsNULL
# How many Passed
# Summary Stats
# Visualize each column

In [4]:
# Train Test Split

features = [
    "attendence",
    "study_hours",
    "previous_score",
    "assignment_completed"
]

target = "passed"

X = df[features]

y = df[target]

X_train,X_test,y_train,y_test= train_test_split(
    X,y,
    test_size=0.2,
    random_state=182
)

print(X_train.shape)
print(y_train.shape)
print(X_test.shape)
print(y_test.shape)

(1600, 4)
(1600,)
(400, 4)
(400,)


In [5]:
models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression())
    ]),
    "Decision Tree" : DecisionTreeClassifier(max_depth = 4),
    "Random Forest" : RandomForestClassifier(n_estimators = 100,max_depth = 5),
    "SVM" : Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),
    "KNN" : KNeighborsClassifier(n_neighbors =3)
}

results = []

trained_models = {}

for model_name,model in models.items():
    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    results.append({
        "model": model_name,
        "accuracy": accuracy,
        "confusion_matrix": confusion_matrix(y_test,y_pred).tolist(),
        "classification_report": classification_report(y_test,y_pred,output_dict=True)
    })
    trained_models[model_name] = model

results_df = pd.DataFrame(results)

results_df.head()

,model,accuracy,confusion_matrix,classification_report
0,Logistic Regression,0.8425,"[[275, 19], [44, 62]]","{'0': {'precision': 0.8620689655172413, 'recal..."
1,Decision Tree,0.8025,"[[264, 30], [49, 57]]","{'0': {'precision': 0.8434504792332268, 'recal..."
2,Random Forest,0.8375,"[[278, 16], [49, 57]]","{'0': {'precision': 0.8501529051987767, 'recal..."
3,SVM,0.8400,"[[279, 15], [49, 57]]","{'0': {'precision': 0.850609756097561, 'recall..."
4,KNN,0.7975,"[[264, 30], [51, 55]]","{'0': {'precision': 0.8380952380952381, 'recal..."


In [7]:
# Experiment Record for Experiment Tracking
# Exp ID : 007
# Model : Decision Tree
# Depth : 5
# Seed : 182
# Acc Score :
# F1-Score :
# Model Location

In [10]:
from pathlib import Path
import os
import mlflow

# Enable local file store backend bypass
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"

# Define a local directory relative to your notebook, then resolve to absolute
TRACKING_DIR = Path("./mlflow_tracking").resolve()
TRACKING_DIR.mkdir(exist_ok=True, parents=True)

mlflow.set_tracking_uri(TRACKING_DIR.as_uri())
mlflow.set_experiment("Student Performance Prediction")

<Experiment: artifact_location='file:///Users/mukundanramesh/projects/ai_ml_course_bootcamp_practice/ai-ml-course-bootcamp/MLOPS/mlflow_tracking/820441688299189998', creation_time=1794393098088, effective_trace_archival_retention=None, experiment_id='820441688299189998', last_update_time=1794393098088, lifecycle_stage='active', name='Student Performance Prediction', tags={}, trace_location=None, workspace='default'>

In [11]:
experiment_models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression())
    ]),
    "Decision Tree Depth 4" : DecisionTreeClassifier(max_depth = 4),
    "Decision Tree Depth 8" : DecisionTreeClassifier(max_depth = 8),
    "Random Forest 50" : RandomForestClassifier(n_estimators = 50,max_depth = 5),
    "Random Forest 100" : RandomForestClassifier(n_estimators = 100,max_depth = 5),
    "SVM" : Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVC())
    ]),
    "KNN" : KNeighborsClassifier(n_neighbors =3)
}


for model_name,model in experiment_models.items():

  with mlflow.start_run(run_name = model_name):

    model.fit(X_train,y_train)
    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test,y_pred)
    f1_score = classification_report(y_test,y_pred,output_dict=True)["weighted avg"]["f1-score"]

    mlflow.log_param("seed",SEED)
    mlflow.log_param("model",model_name)
    mlflow.log_metric("accuracy",accuracy)
    mlflow.log_metric("f1-score",f1_score)

    # Explicitly trust the internal Tree, metric, and KD-Tree structures for safe local serialization
    mlflow.sklearn.log_model(
        model,
        name="model",
        serialization_format="skops",
        skops_trusted_types=[
            "sklearn.tree._tree.Tree",
            "sklearn.metrics._dist_metrics.EuclideanDistance64",
            "sklearn.neighbors._kd_tree.KDTree"
        ]
    )

    print(model_name,accuracy)

Logistic Regression 0.8425
Decision Tree Depth 4 0.8025
Decision Tree Depth 8 0.8075
Random Forest 50 0.825
Random Forest 100 0.8375
SVM 0.84
KNN 0.7975


In [12]:
experiments = mlflow.get_experiment_by_name("Student Performance Prediction")

runs = mlflow.search_runs(
    experiment_ids = experiments.experiment_id,
    order_by = ["metrics.accuracy DESC"]
)

display(runs.head())

best_run_id = runs.iloc[0].run_id

print(best_run_id)



,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.f1-score,metrics.accuracy,params.model,params.seed,tags.mlflow.source.type,tags.mlflow.runName,tags.mlflow.user,tags.mlflow.source.name
0,43020b77eecb4f2bbe52b039e7fd4f5d,820441688299189998,FINISHED,file:///Users/mukundanramesh/projects/ai_ml_co...,2026-11-11 10:32:35.068000+00:00,2026-11-11 10:32:44.708000+00:00,0.835184,0.8425,Logistic Regression,42,NOTEBOOK,Logistic Regression,mukundanramesh,Student Grade Prediction.ipynb
1,49aafd3f1795431caa1b439ee8f2967c,820441688299189998,FINISHED,file:///Users/mukundanramesh/projects/ai_ml_co...,2026-11-11 10:33:00.914000+00:00,2026-11-11 10:33:04.701000+00:00,0.829092,0.8400,SVM,42,NOTEBOOK,SVM,mukundanramesh,Student Grade Prediction.ipynb
2,9e459f30882548c3a786fcb6941ba8e6,820441688299189998,FINISHED,file:///Users/mukundanramesh/projects/ai_ml_co...,2026-11-11 10:32:56.552000+00:00,2026-11-11 10:33:00.913000+00:00,0.825998,0.8375,Random Forest 100,42,NOTEBOOK,Random Forest 100,mukundanramesh,Student Grade Prediction.ipynb
3,c6bcb95ecf374c859264b229eb86e194,820441688299189998,FINISHED,file:///Users/mukundanramesh/projects/ai_ml_co...,2026-11-11 10:32:52.903000+00:00,2026-11-11 10:32:56.551000+00:00,0.813069,0.8250,Random Forest 50,42,NOTEBOOK,Random Forest 50,mukundanramesh,Student Grade Prediction.ipynb
4,d96e67246904467c930f5ee12b40104b,820441688299189998,FINISHED,file:///Users/mukundanramesh/projects/ai_ml_co...,2026-11-11 10:32:48.724000+00:00,2026-11-11 10:32:52.902000+00:00,0.807788,0.8075,Decision Tree Depth 8,42,NOTEBOOK,Decision Tree Depth 8,mukundanramesh,Student Grade Prediction.ipynb


43020b77eecb4f2bbe52b039e7fd4f5d
